# Tema Programare: Construirea Modelelor de Clustering de la Zero

Bun venit la tema despre Algoritmi de Clustering.

Metodele de clustering se numara printre cele mai populare si mai puternice tehnici de machine learning nesupervizat. Ele sunt folosite pentru a descoperi structura ascunsa din date, pentru a grupa puncte similare si pentru a explora seturi de date fara etichete.

In aceasta tema, vei implementa algoritmi de clustering folosind atat scikit-learn, cat si implementari de la zero. Vei lucra cu K-Means, Gaussian Mixture Models, clustering ierarhic si metrici de evaluare, apoi vei implementa versiunile interne ale unora dintre acesti algoritmi.

**Ce vei face in aceasta tema**

* Vei intelege ideile de baza ale clustering-ului nesupervizat
* Vei aplica K-Means si vei interpreta inertia si scorurile silhouette
* Vei selecta valoarea optima a lui K folosind Elbow si Silhouette
* Vei lucra cu Gaussian Mixture Models si asignari probabilistice
* Vei explora clustering ierarhic si metode de linkage
* Vei evalua clustering-ul cu metrici interne si externe
* Vei implementa K-Means de la zero
* Vei implementa GMM de la zero cu algoritmul EM

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul. **Nu adauga si nu modifica cod in afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, asa ca nu te baza pe ele pentru solutia finala. Foloseste spatiile deja oferite in notebook.

* Evita variabilele globale daca nu sunt absolut necesare. Evaluatorul iti testeaza codul intr-un mediu izolat, fara sa ruleze neaparat toate celulele de sus in jos. Din acest motiv, variabilele globale pot sa nu fie disponibile in momentul evaluarii. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Inainte sa trimiti notebook-ul pentru evaluare, salveaza-l mai intai folosind iconita 💾 din coltul stanga-sus al paginii, apoi apasa butonul `Submit assignment` din coltul dreapta-sus.
---


## Cuprins

1. [Introducere in clustering](#1)
2. [Clustering K-Means](#2)
   - [Exercitiul 1: Implementeaza clustering K-Means](#ex-1)
3. [Selectarea lui K optim](#3)
   - [Exercitiul 2: Metoda Elbow si analiza Silhouette](#ex-2)
4. [Gaussian Mixture Models (GMM)](#4)
   - [Exercitiul 3: Implementeaza clustering GMM](#ex-3)
5. [Clustering ierarhic](#5)
   - [Exercitiul 4: Construieste dendrograma si compara strategiile](#ex-4)
6. [Evaluarea clustering-ului](#6)
   - [Exercitiul 5: Aplica metrici pentru clustering](#ex-5)
7. [Implementari de la zero](#7)
   - [Exercitiul 6: K-Means de la zero](#ex-6)
   - [Exercitiul 7: GMM de la zero](#ex-7)
8. [Concluzie](#8)


<a name='1'></a>
## 1 - Introducere in clustering

**Clustering-ul** este o tehnica de invatare nesupervizata care grupeaza puncte de date similare fara a folosi etichete. Este folosita frecvent pentru:

- **Segmentarea clientilor**
- **Detectarea anomaliilor**
- **Compresie si rezumare de date**
- **Explorare vizuala**


### Importa bibliotecile necesare


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.datasets import make_blobs, make_moons, load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score, normalized_mutual_info_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("All libraries imported successfully!")

### Functii ajutatoare


Vom defini cateva functii ajutatoare pentru vizualizare si evaluare.


In [ ]:
def plot_clusters(X, labels, centers=None, title="Cluster Assignments"):
    """Plot 2D clustering results."""
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', 
                         alpha=0.6, edgecolors='k', s=50)
    
    if centers is not None:
        plt.scatter(centers[:, 0], centers[:, 1], 
                   c='red', marker='X', s=300, 
                   edgecolors='black', linewidths=2, 
                   label='Centroids')
        plt.legend()
    
    plt.colorbar(scatter, label='Cluster')
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.tight_layout()
    plt.show()


def plot_silhouette(X, labels, n_clusters):
    """Plot silhouette analysis."""
    silhouette_vals = silhouette_samples(X, labels)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    y_lower = 10
    
    for i in range(n_clusters):
        cluster_silhouette_vals = silhouette_vals[labels == i]
        cluster_silhouette_vals.sort()
        
        size_cluster_i = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.viridis(float(i) / n_clusters)
        ax.fill_betweenx(np.arange(y_lower, y_upper),
                         0, cluster_silhouette_vals,
                         facecolor=color, edgecolor=color, alpha=0.7)
        
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    
    ax.set_xlabel("Silhouette Coefficient")
    ax.set_ylabel("Cluster")
    ax.axvline(x=silhouette_score(X, labels), color="red", linestyle="--", 
               label=f"Average: {silhouette_score(X, labels):.3f}")
    ax.set_title(f"Silhouette Plot for {n_clusters} Clusters")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_gmm_ellipses(gmm, X, labels):
    """Plot GMM clusters with confidence ellipses."""
    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, gmm.n_components))
    
    for i in range(gmm.n_components):
        # Plot points
        mask = labels == i
        plt.scatter(X[mask, 0], X[mask, 1], c=[colors[i]], 
                   alpha=0.4, s=30, edgecolors='k', linewidths=0.5)
        
        # Plot ellipse
        mean = gmm.means_[i]
        covar = gmm.covariances_[i]
        
        v, w = np.linalg.eigh(covar)
        v = 2.0 * np.sqrt(2.0) * np.sqrt(v)
        u = w[0] / np.linalg.norm(w[0])
        
        angle = np.arctan(u[1] / u[0])
        angle = 180.0 * angle / np.pi
        
        ell = plt.matplotlib.patches.Ellipse(
            mean, v[0], v[1], angle=180.0 + angle, 
            color=colors[i], alpha=0.3, linewidth=2
        )
        plt.gca().add_patch(ell)
        plt.scatter(mean[0], mean[1], c=[colors[i]], marker='X', 
                   s=200, edgecolors='black', linewidths=2)
    
    plt.title("GMM Clustering with Confidence Ellipses")
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.tight_layout()
    plt.show()

print("Helper functions defined!")

### Incarca si pregateste datele


In [ ]:
# Create synthetic dataset with clear clusters
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, 
                              n_features=2, cluster_std=0.60, 
                              random_state=42)

# Create non-convex dataset (moons)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# Load Iris dataset for evaluation
iris = load_iris()
X_iris = iris.data[:, [2, 3]]  # Use petal dimensions
y_iris = iris.target

# Standardize features
scaler = StandardScaler()
X_blobs_scaled = scaler.fit_transform(X_blobs)
X_moons_scaled = scaler.fit_transform(X_moons)
X_iris_scaled = scaler.fit_transform(X_iris)

print(f"Blobs dataset shape: {X_blobs.shape}")
print(f"Moons dataset shape: {X_moons.shape}")
print(f"Iris dataset shape: {X_iris.shape}")
print(f"\nIris has {len(np.unique(y_iris))} true classes")

<a name='2'></a>
## 2 - Clustering K-Means

**K-Means** este unul dintre cei mai populari algoritmi de clustering. El partitioneaza datele in $K$ clustere prin:

1. **Initializare**: selecteaza aleator $K$ centroizi;
2. **Atribuire**: aloca fiecare punct celui mai apropiat centroid;
3. **Actualizare**: recalculeaza centroizii ca medie a punctelor din fiecare cluster;
4. **Repetare** pana la convergenta.


<a name='ex-1'></a>
### Exercitiul 1 - K-Means Clustering

**Sarcina ta:**

Implementeaza functia `apply_kmeans`, care:
1. aplica K-Means pe date;
2. calculeaza si returneaza inertia;
3. calculeaza scorul silhouette;
4. returneaza etichetele clusterelor si modelul antrenat.


In [ ]:
# GRADED FUNCTION: apply_kmeans
def apply_kmeans(X, n_clusters=4, random_state=42):
    """
    Apply K-Means clustering to data.
    
    Args:
        X: Feature matrix
        n_clusters: Number of clusters
        random_state: Random seed
    
    Returns:
        dict with keys: 'labels', 'centers', 'inertia', 'silhouette'
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 8 lines)
    # Create KMeans object
    kmeans = None
    
    # Fit to data
    
    # Get cluster labels
    labels = None
    
    # Get cluster centers
    centers = None
    
    # Get inertia
    inertia = None
    
    # Compute silhouette score
    sil_score = None
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'labels': labels,
        'centers': centers,
        'inertia': inertia,
        'silhouette': sil_score
    }

In [ ]:
# Test K-Means on blobs dataset
result = apply_kmeans(X_blobs_scaled, n_clusters=4)

print(f"K-Means Results (K=4):")
print(f"Inertia: {result['inertia']:.4f}")
print(f"Silhouette Score: {result['silhouette']:.4f}")
print(f"\nCluster sizes: {np.bincount(result['labels'])}")

# Visualize
plot_clusters(X_blobs_scaled, result['labels'], result['centers'],
              title="K-Means Clustering (K=4)")

##### **Rezultatul asteptat**


In [ ]:
unittests.exercise_1(apply_kmeans)

<a name='3'></a>
## 3 - Selectarea lui K optim

Alegerea numarului corect de clustere ($K$) este esentiala, dar dificila. Doua metode populare sunt:

### 1. Metoda Elbow
Se traseaza inertia in functie de $K$ si se cauta un "cot" in curba.

### 2. Analiza Silhouette
Evalueaza cat de bine este separat fiecare cluster fata de celelalte.


<a name='ex-2'></a>
### Exercitiul 2 - Metoda Elbow si analiza Silhouette

**Sarcina ta:**

Implementeaza functia `find_optimal_k`, care:
1. testeaza valori diferite pentru $K$;
2. calculeaza inertia si scorul silhouette;
3. identifica o valoare buna pentru $K$;
4. returneaza rezultatele pentru analiza.


In [ ]:
# GRADED FUNCTION: find_optimal_k
def find_optimal_k(X, max_k=10):
    """
    Find optimal K using elbow method and silhouette analysis.
    
    Args:
        X: Feature matrix
        max_k: Maximum K to test
    
    Returns:
        optimal_k: Best K value based on silhouette score
    """
    inertias = []
    silhouettes = []
    k_range = range(2, max_k + 1)
    
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    # Loop over K values
    for k in k_range:
        # Apply K-Means
        result = None
        
        # Store inertia
        
        # Store silhouette score
        
    ### SFÂRȘIT DE COD AICI ###
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Elbow plot
    ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Clusters (K)', fontsize=12)
    ax1.set_ylabel('Inertia', fontsize=12)
    ax1.set_title('Elbow Method', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Silhouette plot
    ax2.plot(k_range, silhouettes, 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Clusters (K)', fontsize=12)
    ax2.set_ylabel('Silhouette Score', fontsize=12)
    ax2.set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Mark optimal K
    optimal_k = k_range[np.argmax(silhouettes)]
    ax2.axvline(x=optimal_k, color='green', linestyle='--', linewidth=2,
                label=f'Optimal K={optimal_k}')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    return optimal_k

In [ ]:
# Find optimal K for blobs dataset
optimal_k = find_optimal_k(X_blobs_scaled, max_k=8)
print(f"\nOptimal K (based on silhouette): {optimal_k}")

# Apply with optimal K
result_optimal = apply_kmeans(X_blobs_scaled, n_clusters=optimal_k)
print(f"\nOptimal Clustering Results:")
print(f"Silhouette Score: {result_optimal['silhouette']:.4f}")

# Visualize silhouette plot
plot_silhouette(X_blobs_scaled, result_optimal['labels'], optimal_k)

##### **Rezultatul asteptat**


In [ ]:
unittests.exercise_2(find_optimal_k)

<a name='4'></a>
## 4 - Gaussian Mixture Models (GMM)

**Gaussian Mixture Models** sunt algoritmi probabilistici de clustering care presupun ca datele provin dintr-un amestec de distributii Gaussiene.

### Diferente-cheie fata de K-Means:
- GMM ofera probabilitati de apartenenta la cluster;
- poate modela clustere eliptice;
- foloseste o abordare probabilistica, nu doar distante fata de centroid.


<a name='ex-3'></a>
### Exercitiul 3 - Implementeaza clustering GMM

**Sarcina ta:**

Implementeaza functia `apply_gmm`, care:
1. creeaza si antreneaza un Gaussian Mixture Model;
2. prezice asignarile de cluster;
3. calculeaza BIC, AIC si scorul silhouette;
4. returneaza si probabilitatile de apartenenta la cluster.


In [ ]:
# GRADED FUNCTION: apply_gmm
def apply_gmm(X, n_components=4, random_state=42):
    """
    Apply Gaussian Mixture Model clustering.
    
    Args:
        X: Feature matrix
        n_components: Number of mixture components
        random_state: Random seed
    
    Returns:
        dict with keys: 'labels', 'probabilities', 'gmm', 'bic', 'aic', 'silhouette'
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 10 lines)
    # Create GMM object
    gmm = None
    
    # Fit to data
    
    # Get cluster labels (hard assignment)
    labels = None
    
    # Get cluster probabilities (soft assignment)
    probabilities = None
    
    # Compute BIC and AIC
    bic = None
    aic = None
    
    # Compute silhouette score
    sil_score = None
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'labels': labels,
        'probabilities': probabilities,
        'gmm': gmm,
        'bic': bic,
        'aic': aic,
        'silhouette': sil_score
    }

In [ ]:
# Test GMM on blobs dataset
gmm_result = apply_gmm(X_blobs_scaled, n_components=4)

print(f"GMM Results (K=4):")
print(f"BIC: {gmm_result['bic']:.4f}")
print(f"AIC: {gmm_result['aic']:.4f}")
print(f"Silhouette Score: {gmm_result['silhouette']:.4f}")
print(f"\nCluster sizes: {np.bincount(gmm_result['labels'])}")

# Show probability distribution for first 5 points
print(f"\nCluster probabilities for first 5 points:")
print(gmm_result['probabilities'][:5])

# Visualize with confidence ellipses
plot_gmm_ellipses(gmm_result['gmm'], X_blobs_scaled, gmm_result['labels'])

##### **Rezultatul asteptat**


In [ ]:
unittests.exercise_3(apply_gmm)

<a name='5'></a>
## 5 - Clustering ierarhic

**Clustering-ul ierarhic** construieste un arbore al clusterelor (dendrograma) fara sa fie nevoie sa specifici de la inceput valoarea lui $K$.

### Doua abordari:
1. **Aglomerativa (Bottom-Up)**: porneste cu fiecare punct separat si le uneste treptat.
2. **Diviziva (Top-Down)**: porneste cu toate punctele intr-un singur cluster si le separa progresiv.


<a name='ex-4'></a>
### Exercitiul 4 - Construieste dendrograma si compara strategiile

**Sarcina ta:**

Implementeaza doua functii:
1. `apply_hierarchical`: aplica clustering aglomerativ cu metoda de legatura specificata;
2. `compare_linkage_methods`: compara toate strategiile de linkage si returneaza rezultatele.


In [ ]:
# GRADED FUNCTION: apply_hierarchical
def apply_hierarchical(X, n_clusters=4, linkage_method='ward'):
    """
    Apply agglomerative hierarchical clustering.
    
    Args:
        X: Feature matrix
        n_clusters: Number of clusters
        linkage_method: 'ward', 'complete', 'average', or 'single'
    
    Returns:
        dict with keys: 'labels', 'silhouette'
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    # Create AgglomerativeClustering object
    agg_clustering = None
    
    # Fit and predict
    labels = None
    
    # Compute silhouette score
    sil_score = None
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'labels': labels,
        'silhouette': sil_score
    }

In [ ]:
# Apply hierarchical clustering with Ward linkage
hier_result = apply_hierarchical(X_blobs_scaled, n_clusters=4, linkage_method='ward')

print(f"Hierarchical Clustering (Ward linkage):")
print(f"Silhouette Score: {hier_result['silhouette']:.4f}")
print(f"Cluster sizes: {np.bincount(hier_result['labels'])}")

# Visualize
plot_clusters(X_blobs_scaled, hier_result['labels'],
              title="Hierarchical Clustering (Ward)")

# Plot dendrogram
plt.figure(figsize=(12, 6))
Z = linkage(X_blobs_scaled, method='ward')
dendrogram(Z, truncate_mode='lastp', p=30)
plt.title('Hierarchical Clustering Dendrogram (Ward)', fontsize=14, fontweight='bold')
plt.xlabel('Sample Index (or Cluster Size)', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_4(apply_hierarchical)

#### `compare_linkage_methods` - Compara toate strategiile de linkage

Implementeaza `compare_linkage_methods`, care aplica `apply_hierarchical` pentru fiecare dintre cele patru metode de linkage (`'ward'`, `'complete'`, `'average'`, `'single'`) si compara scorurile lor.


In [ ]:
# GRADED FUNCTION: compare_linkage_methods
def compare_linkage_methods(X, n_clusters=4):
    """
    Compare different linkage methods for hierarchical clustering.
    
    Args:
        X: Feature matrix
        n_clusters: Number of clusters
    
    Returns:
        results: dict mapping linkage method to results
    """
    linkages = ['ward', 'complete', 'average', 'single']
    results = {}
    
    ### ÎNCEPUT DE COD AICI ### (≈ 3 lines)
    for linkage_method in linkages:
        # Apply hierarchical clustering
        result = None
        # Store result
        
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Compare linkage methods
comparison = compare_linkage_methods(X_blobs_scaled, n_clusters=4)

print("Comparison of Linkage Methods:\n")
print(f"{'Method':<12} {'Silhouette':<12}")
print("-" * 24)
for method, result in comparison.items():
    print(f"{method:<12} {result['silhouette']:<12.4f}")

# Visualize all methods
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, (method, result) in enumerate(comparison.items()):
    ax = axes[idx]
    scatter = ax.scatter(X_blobs_scaled[:, 0], X_blobs_scaled[:, 1], 
                        c=result['labels'], cmap='viridis', 
                        alpha=0.6, edgecolors='k', s=50)
    ax.set_title(f"{method.capitalize()} Linkage\nSilhouette: {result['silhouette']:.3f}",
                fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    plt.colorbar(scatter, ax=ax, label='Cluster')

plt.tight_layout()
plt.show()

##### **Rezultatul asteptat**


In [ ]:
unittests.exercise_5(compare_linkage_methods)

<a name='6'></a>
## 6 - Evaluarea clustering-ului

Evaluarea calitatii clustering-ului este dificila atunci cand nu avem etichete reale. Folosim doua tipuri de metrici:

### 1. Metrici interne
- **Silhouette Score**
- **Davies-Bouldin Index**
- **Calinski-Harabasz Score**

### 2. Metrici externe
Se folosesc doar atunci cand avem etichete reale pentru comparatie.


<a name='ex-5'></a>
### Exercitiul 5 - Aplica metrici pentru clustering

**Sarcina ta:**

Implementeaza functia `evaluate_clustering`, care:
1. calculeaza metrici interne;
2. daca exista etichete reale, calculeaza si metrici externe;
3. returneaza rezultatele intr-un format usor de comparat.


In [ ]:
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

# GRADED FUNCTION: evaluate_clustering
def evaluate_clustering(X, labels, y_true=None):
    """
    Evaluate clustering using internal and external metrics.
    
    Args:
        X: Feature matrix
        labels: Predicted cluster labels
        y_true: True labels (optional, for external metrics)
    
    Returns:
        dict with evaluation metrics
    """
    metrics = {}
    
    ### ÎNCEPUT DE COD AICI ### (≈ 10 lines)
    # Internal metrics
    metrics['silhouette'] = None
    metrics['davies_bouldin'] = None  # Lower is better
    metrics['calinski_harabasz'] = None  # Higher is better
    
    # External metrics (if true labels available)
    if y_true is not None:
        metrics['ari'] = None
        metrics['nmi'] = None
    ### SFÂRȘIT DE COD AICI ###
    
    return metrics

In [ ]:
# Evaluate different algorithms on Iris dataset
print("=" * 60)
print("CLUSTERING EVALUATION ON IRIS DATASET")
print("=" * 60)

# Apply different clustering methods
kmeans_iris = apply_kmeans(X_iris_scaled, n_clusters=3)
gmm_iris = apply_gmm(X_iris_scaled, n_components=3)
hier_iris = apply_hierarchical(X_iris_scaled, n_clusters=3)

# Evaluate each method
methods = {
    'K-Means': kmeans_iris['labels'],
    'GMM': gmm_iris['labels'],
    'Hierarchical': hier_iris['labels']
}

results_df = []
for name, labels in methods.items():
    metrics = evaluate_clustering(X_iris_scaled, labels, y_iris)
    metrics['Method'] = name
    results_df.append(metrics)

# Create comparison table
df = pd.DataFrame(results_df)
df = df[['Method', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'ari', 'nmi']]

print("\n")
print(df.to_string(index=False))
print("\n")

# Find best method for each metric
print("Best Performers:")
print(f"  Highest Silhouette: {df.loc[df['silhouette'].idxmax(), 'Method']} ({df['silhouette'].max():.4f})")
print(f"  Lowest Davies-Bouldin: {df.loc[df['davies_bouldin'].idxmin(), 'Method']} ({df['davies_bouldin'].min():.4f})")
print(f"  Highest Calinski-Harabasz: {df.loc[df['calinski_harabasz'].idxmax(), 'Method']} ({df['calinski_harabasz'].max():.2f})")
print(f"  Highest ARI: {df.loc[df['ari'].idxmax(), 'Method']} ({df['ari'].max():.4f})")
print(f"  Highest NMI: {df.loc[df['nmi'].idxmax(), 'Method']} ({df['nmi'].max():.4f})")

##### **Rezultatul asteptat**


In [ ]:
unittests.exercise_6(evaluate_clustering)

<a name='7'></a>
## 7 - Implementari de la zero

Intelegerea interiorului algoritmilor de clustering te ajuta sa:
- ii adaptezi pentru nevoi specifice;
- depanezi cazurile in care esueaza;
- optimizezi performanta si stabilitatea lor.


<a name='ex-6'></a>
### Exercitiul 6 - K-Means de la zero

**Sarcina ta:**

Implementeaza algoritmul K-Means de la zero:
1. `initialize_centroids`: selecteaza aleator puncte initiale;
2. `assign_clusters`: atribuie fiecare punct celui mai apropiat centroid;
3. `update_centroids`: recalculeaza centroizii;
4. `kmeans_from_scratch`: combina pasii intr-o bucla completa pana la convergenta.


In [ ]:
# GRADED FUNCTION: initialize_centroids
def initialize_centroids(X, k, random_state=42):
    """
    Initialize centroids by randomly selecting k data points.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        k: Number of clusters
        random_state: Random seed
    
    Returns:
        centroids: Initial centroids (k, n_features)
    """
    np.random.seed(random_state)
    
    ### ÎNCEPUT DE COD AICI ### (≈ 2 lines)
    # Randomly select k indices
    indices = None
    # Get corresponding points
    centroids = None
    ### SFÂRȘIT DE COD AICI ###
    
    return centroids

In [ ]:
unittests.exercise_7(initialize_centroids)

#### `assign_clusters` - Atribuie fiecare punct celui mai apropiat centroid

Calculeaza distanta Euclidiana de la fiecare punct la fiecare centroid si atribuie fiecare punct clusterului cu cel mai apropiat centroid.


In [ ]:
# GRADED FUNCTION: assign_clusters
def assign_clusters(X, centroids):
    """
    Assign each point to the nearest centroid.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        centroids: Current centroids (k, n_features)
    
    Returns:
        labels: Cluster assignment for each point (n_samples,)
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 3 lines)
    # Compute distances from each point to each centroid
    distances = None
    # Assign to nearest centroid
    labels = None
    ### SFÂRȘIT DE COD AICI ###
    
    return labels

In [ ]:
unittests.exercise_8(assign_clusters)

#### `update_centroids` - Recalculeaza centroizii

Recalculeaza fiecare centroid ca media tuturor punctelor atribuite clusterului respectiv. Gestioneaza corect si cazul clusterelor goale.


In [ ]:
# GRADED FUNCTION: update_centroids
def update_centroids(X, labels, k):
    """
    Update centroids as the mean of assigned points.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        labels: Current cluster assignments (n_samples,)
        k: Number of clusters
    
    Returns:
        centroids: Updated centroids (k, n_features)
    """
    n_features = X.shape[1]
    centroids = np.zeros((k, n_features))
    
    ### ÎNCEPUT DE COD AICI ### (≈ 4 lines)
    for cluster in range(k):
        # Get points in this cluster
        cluster_points = None
        # Compute mean (handle empty clusters)
        if len(cluster_points) > 0:
            centroids[cluster] = None
    ### SFÂRȘIT DE COD AICI ###
    
    return centroids

In [ ]:
unittests.exercise_9(update_centroids)

#### `kmeans_from_scratch` - Bucla completa K-Means

Combina cei trei pasi de mai sus in algoritmul complet K-Means: initializeaza centroizii, apoi alterneaza intre atribuire si actualizare pana la convergenta sau pana cand se atinge `max_iters`.


In [ ]:
# GRADED FUNCTION: kmeans_from_scratch
def kmeans_from_scratch(X, k, max_iters=100, random_state=42):
    """
    K-Means clustering from scratch.
    
    Args:
        X: Feature matrix
        k: Number of clusters
        max_iters: Maximum iterations
        random_state: Random seed
    
    Returns:
        dict with 'labels', 'centroids', 'inertia', 'n_iters'
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 15 lines)
    # Initialize centroids
    centroids = None
    
    # Main loop
    for iteration in range(max_iters):
        # Assignment step
        labels = None
        
        # Update step
        new_centroids = None
        
        # Check convergence
        if np.allclose(centroids, new_centroids):
            centroids = new_centroids
            break
        
        centroids = new_centroids
    
    # Compute final inertia
    distances = None
    min_distances = None
    inertia = None
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'labels': labels,
        'centroids': centroids,
        'inertia': inertia,
        'n_iters': iteration + 1
    }

In [ ]:
unittests.exercise_10(kmeans_from_scratch)

In [ ]:
# Test K-Means from scratch
scratch_result = kmeans_from_scratch(X_blobs_scaled, k=4, random_state=42)

print("K-Means from Scratch:")
print(f"Converged in {scratch_result['n_iters']} iterations")
print(f"Inertia: {scratch_result['inertia']:.4f}")
print(f"Cluster sizes: {np.bincount(scratch_result['labels'])}")

# Compare with sklearn
sklearn_result = apply_kmeans(X_blobs_scaled, n_clusters=4)
print(f"\nSklearn K-Means Inertia: {sklearn_result['inertia']:.4f}")
print(f"Difference: {abs(scratch_result['inertia'] - sklearn_result['inertia']):.6f}")

# Visualize
plot_clusters(X_blobs_scaled, scratch_result['labels'], 
              scratch_result['centroids'],
              title="K-Means from Scratch")

##### **Rezultatul asteptat**


<a name='ex-7'></a>
### Exercitiul 7 - GMM de la zero (Expectation-Maximization)

**Sarcina ta:**

Implementeaza un Gaussian Mixture Model folosind algoritmul Expectation-Maximization (EM).

**E-Step**: calculeaza probabilitatile de apartenenta la fiecare componenta.

**M-Step**: actualizeaza mediile, covariantele si ponderile de amestec.


In [ ]:
# GRADED FUNCTION: gmm_from_scratch
def gmm_from_scratch(X, k, max_iters=100, tol=1e-4, random_state=42):
    """
    Gaussian Mixture Model using Expectation-Maximization.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        k: Number of components
        max_iters: Maximum iterations
        tol: Convergence tolerance
        random_state: Random seed
    
    Returns:
        dict with 'labels', 'means', 'covariances', 'weights', 'responsibilities'
    """
    n_samples, n_features = X.shape
    
    ### ÎNCEPUT DE COD AICI ### (≈ 40 lines)
    # Initialize using K-Means
    kmeans_result = None
    means = None
    
    # Initialize covariances as spherical
    covariances = np.array([np.eye(n_features) for _ in range(k)])
    
    # Initialize weights (mixing coefficients)
    weights = None
    
    prev_log_likelihood = -np.inf
    
    for iteration in range(max_iters):
        # E-Step: Compute responsibilities
        responsibilities = np.zeros((k, n_samples))
        
        for j in range(k):
            # Compute probability density for cluster j
            responsibilities[j] = None
        
        # Normalize responsibilities
        responsibilities /= responsibilities.sum(axis=0)
        
        # M-Step: Update parameters
        for j in range(k):
            # Total responsibility for cluster j
            Nj = None
            
            # Update weights
            weights[j] = None
            
            # Update means
            means[j] = None
            
            # Update covariances
            diff = X - means[j]
            covariances[j] = None
            # Add small value to diagonal for numerical stability
            covariances[j] += np.eye(n_features) * 1e-6
        
        # Compute log likelihood
        log_likelihood = 0
        for j in range(k):
            log_likelihood += (weights[j] * 
                             multivariate_normal.pdf(X, mean=means[j], 
                                                    cov=covariances[j]))
        log_likelihood = np.log(log_likelihood).sum()
        
        # Check convergence
        if abs(log_likelihood - prev_log_likelihood) < tol:
            break
        prev_log_likelihood = log_likelihood
    
    # Get final cluster assignments
    labels = None
    ### SFÂRȘIT DE COD AICI ###
    
    return {
        'labels': labels,
        'means': means,
        'covariances': covariances,
        'weights': weights,
        'responsibilities': responsibilities,
        'n_iters': iteration + 1
    }

In [ ]:
unittests.exercise_11(gmm_from_scratch)

In [ ]:
# Test GMM from scratch
gmm_scratch = gmm_from_scratch(X_blobs_scaled, k=4, random_state=42)

print("GMM from Scratch:")
print(f"Converged in {gmm_scratch['n_iters']} iterations")
print(f"Mixing weights: {gmm_scratch['weights']}")
print(f"Cluster sizes: {np.bincount(gmm_scratch['labels'])}")

# Compare with sklearn
gmm_sklearn = apply_gmm(X_blobs_scaled, n_components=4)
print(f"\nComparison with sklearn:")
print(f"Scratch silhouette: {silhouette_score(X_blobs_scaled, gmm_scratch['labels']):.4f}")
print(f"Sklearn silhouette: {gmm_sklearn['silhouette']:.4f}")

# Visualize
plot_clusters(X_blobs_scaled, gmm_scratch['labels'],
              gmm_scratch['means'],
              title="GMM from Scratch")

# Show soft assignments for first 5 points
print(f"\nSoft cluster assignments (first 5 points):")
print(gmm_scratch['responsibilities'][:, :5].T)

##### **Rezultatul asteptat**


<a name='8'></a>
## 8 - Concluzie

**Felicitari!** Ai finalizat tema despre Algoritmi de Clustering!

### Ce ai invatat:

1. **K-Means Clustering**: ai implementat clustering partitional si ai analizat inertia si scorul silhouette
2. **Selectarea lui K optim**: ai folosit metodele Elbow si Silhouette pentru a alege numarul de clustere
3. **Gaussian Mixture Models**: ai lucrat cu clustering probabilistic si probabilitati de apartenenta
4. **Clustering ierarhic**: ai comparat strategii de linkage si ai interpretat dendrograme
5. **Evaluarea clustering-ului**: ai aplicat metrici interne si externe
6. **Implementari de la zero**: ai construit K-Means si GMM fara scikit-learn
